# Observable class - Lsstypes implementation

In [ ]:
# Setup
from helpers import make_file

from acm import setup_logging

setup_logging()

data = make_file("observable.h5", backend="lsstypes")

The `LsstypesObservable` class handles observables stored in a `lsstypes.ObservableTree` object. It is a subclass of `BaseObservable` and implements the same interface, so it can be used interchangeably with other observable types.

The data objects (`x`, `y`, ...) are stored as sub-objects of type `lsstypes.ObservableTree` in the main tree.

The data object store samples in the first layer of the `ObservableTree`, each containing their own structure. When reshaping on 2 dimensional arrays, each sample is flattened to get the second dimension.

### Object requirements
The loaded object must contain at least `x` and `y` sub-trees, indexed by a `name` label. The first-level labels of `x` and `y` must match.

### Choice of backend
**Pros**
- Sparse-indexing allowed in the first layer:
  - Sample indexes can be sparsly indexed, even with several combinations (e.g. cosmology, hod)
  - Nested sample indexes can have different structures - though feature indexes must be identical (by construction of the compressed data). E.g. each cosmology can have a different number of HODs, as the indexing is done by (cosmology, hod) pairs instead of nested (cosmology, hod) indexes.
- No `NaN` padding: Any `NaN` value found is computational, not structural.
- Precomputed filters: The filtering precomputes a mask from the filters, allowing a fast application of the filters on the 2D predictions instead of casting it to a tree.

**Cons**
- Slower filtering: When calling some properties (e.g. with getattr), the  native filtering is applied on the tree, which is slower than the precomputed mask.
  - This cost is usually paid only once, when the user calls sets the filters. A 2D index mask is cached for later use.
  - It can also be paid when `nested=True`, as this casts the result to `lsstypes`.
- More complex structure: While aligning to the convention of the DESI pipeline, the `lsstype` structure and API is more complex and less documented (for now) than the `xarray`/`numpy` which is fairly well known and documented.

In [ ]:
from acm.observables.lsstypes import LsstypesObservable

obs = LsstypesObservable(data=data)
obs

> Directly passing data to the `LsstypesObservable` class allows missing data elements (no checks are performed on the data structure). 
> This is done on purpose to allow custom data structures for specific cases (e.g. measurements with no `x` value, ...)

In [ ]:
# Accessing the raw ObservableTree through raw=True
raw_y = obs.get_data("y", raw=True)
type(raw_y), raw_y.labels()

> Note: `lsstypes` does not allow slice selection on labels, only coordinates, for index slices (not values). 
> For concistency, the `set_filters` method transforms slices of values to tuple of values to pass to the `lsstypes` selection method (step is not supported and is silently ignored).

In [ ]:
# nested=True preserves the filtered but unflattened tree structure, cast to a nested array
obs.clear_filters()
obs.set_filters(ells=[0, 2])

s1 = obs.get_data("y").shape          # flattened 2D: (n_samples, n_ells * n_k)
s2 = obs.get_data("y", nested=True).shape   # unflattened, mirrors the tree structure

s1, s2

The class also expose several methods to access some specific properties from the filtered tree.

Assuming the same features labels, `get_coordinate_list` resolves either the unique values of said label across the tree (e.g. "ells") or the unique values of a coordinate stored on the leaves themselves (e.g. "k").

In [ ]:
# get_coordinate_list resolves two different things depending on the name:
# - a structural label shared across branches (e.g. "ells")
obs.set_filters(i=[0], ells=[0, 2])
ells = obs.get_coordinate_list("ells")
# - a coordinate stored on the leaves themselves (e.g. "k")
k = obs.get_coordinate_list("k")

ells, k

> Note: The `get_coordinate_list` method returns a unique list of values, which may not preserve the order of the original data.
> To ensure correct ordering (especially on the names of the parameters), the `x_names` property returns the names in the order they are stored in the filtered data.

In [ ]:
obs.x_names, obs.get_coordinate_list("parameters") # Can be selected, try set_filters with parameters=["h0"]

The class exposes `__getattr__` to allow the direct access to the underlying `ObservableTree` attributes (with filters applied), so that the user can use the `lsstypes` API directly on the observable object. For example, `obs.labels()` will return the labels of the underlying tree.

In [ ]:
# __getattr__ exposes the underlying (filtered) ObservableTree API directly
obs.labels(), obs.size

In [ ]:
# Data variable names (x, y, covariance_y, ...) short-circuit straight to get_data
obs.y.shape  # equivalent to obs.get_data("y").shape

`get_test_set` is a shortcut method to access the `x_test` and `y_test` data variables (if they exist) as 2D arrays with filters and selection applied.

In [ ]:
x_test, y_test = obs.get_test_set()
x_test.shape, y_test.shape